# Medicare Benefits Description Generator
### LangChain + Azure OpenAI

**Cell 1** — Converts the two source files into a single JSON payload  
**Cells 2–8** — Consume that JSON end-to-end: parse → build tasks → LLM → CSV output

---
## Cell 1 — Convert source files → single JSON
Reads `input_example.txt` and `ParseBenefitFileRule_BenefitMap_loadid142.csv`,  
merges them into one JSON object, and optionally saves it to disk.

In [ ]:
import csv
import json
import io
import os

# ── File paths ────────────────────────────────────────────────────────────────
PBP_FILE_PATH   = "input_example.txt"
RULES_FILE_PATH = "ParseBenefitFileRule_BenefitMap_loadid142.csv"
LOAD_ID         = 142


def files_to_json(pbp_path: str, rules_path: str, load_id: int) -> dict:
    """
    Read both source files and merge into a single JSON-serialisable dict.

    Output schema:
    {
      "load_id": 142,
      "pbp": [                         # every row of input_example.txt
        {"FileName": ..., "ID": ..., "header": ..., "category": ...,
         "field": ..., "value": ..., "DT": ...},
        ...
      ],
      "benefit_rules": [               # every row of ParseBenefitFileRule_BenefitMap_loadid142.csv
        {"LoadID": ..., "FileName": ..., "ParseBenefitRuleId": ...,
         "RuleOrder": ..., "BenefitId": ..., "NetworkType": ...,
         "TierType": ..., "BenefitName": ..., "DerivedRule": ...,
         "DynamicLongValue": ..., "DynamicShortValue": ...,
         "DynamicTinyValue": ..., "value": ..., "field": ...},
        ...
      ]
    }
    """
    with open(pbp_path, encoding="utf-8-sig") as f:
        pbp_rows = [row for row in csv.DictReader(f) if any(v.strip() for v in row.values())]

    with open(rules_path, encoding="utf-8-sig") as f:
        rules_rows = [row for row in csv.DictReader(f) if any(v.strip() for v in row.values())]

    return {
        "load_id":       load_id,
        "pbp":           pbp_rows,
        "benefit_rules": rules_rows,
    }


# ── Build the JSON ────────────────────────────────────────────────────────────
input_json = files_to_json(PBP_FILE_PATH, RULES_FILE_PATH, LOAD_ID)

# ── Optionally save to disk (comment out if not needed) ───────────────────────
JSON_OUTPUT_PATH = f"medicare_input_loadid{LOAD_ID}.json"
with open(JSON_OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(input_json, f, indent=2)

print(f"✅ JSON built")
print(f"   load_id:        {input_json['load_id']}")
print(f"   pbp rows:       {len(input_json['pbp']):,}")
print(f"   benefit_rules:  {len(input_json['benefit_rules']):,}")
print(f"   JSON size:      {len(json.dumps(input_json)):,} chars")
print(f"   Saved to:       {JSON_OUTPUT_PATH}")

---
## Cell 2 — Install & configure
Installs dependencies and sets Azure OpenAI credentials.

In [ ]:
# !pip install langchain langchain-openai langchain-core openai pandas   # run once

import re
import pandas as pd
from collections import defaultdict
from dataclasses import dataclass, field
from typing import Dict, List, Tuple, Optional

from langchain_openai import AzureChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# ── Azure OpenAI credentials ──────────────────────────────────────────────────
AZURE_OPENAI_ENDPOINT    = os.getenv("AZURE_OPENAI_ENDPOINT",    "https://YOUR-RESOURCE.openai.azure.com/")
AZURE_OPENAI_API_KEY     = os.getenv("AZURE_OPENAI_API_KEY",     "YOUR_API_KEY")
AZURE_OPENAI_DEPLOYMENT  = os.getenv("AZURE_OPENAI_DEPLOYMENT",  "gpt-4o")
AZURE_OPENAI_API_VERSION = os.getenv("AZURE_OPENAI_API_VERSION", "2024-02-01")
CARRIER_PREFIX           = "MOM"

# ── Static lookup tables ──────────────────────────────────────────────────────
SERVICE_TYPE_MAP = {
    0: "N/A",       20: "Tier 1",     21: "Tier 2",    22: "Tier 3",
    23: "Tier 4",   24: "Tier 5",     25: "Tier 6",
    47: "Occupational Therapy",        49: "Cardiac Rehab",
    50: "Diagnostic Procedures",       51: "X-Rays",
    53: "Diagnostic Radiology",        54: "Therapeutic Radiology",
    63: "Prescription Hearing Aid",
    66: "Exam (Diagnose and treatment)", 67: "Exam (Routine)",
    68: "Eyeglasses",  70: "Contacts",  71: "Lenses",  72: "Frames",
    106: "Ambulatory Center",          108: "Cleaning",
    115: "days 1 through 20",          125: "days 1 through 8",
    134: "days 21 through 100",        163: "days 9 through 90",
    167: "Fitting Eval",
    169: "Group Therapy",              170: "Individual Therapy",
    171: "Lab",       173: "Outpatient Hospital",  174: "Outpatient X-ray",
    175: "Part B drugs",               176: "Physical Exam",
    177: "Physical Therapy",           179: "Prosthetic Devices",
    181: "Therapeutic Radiology",      182: "Related Supplies",
    184: "Shoes/Inserts",              185: "Supplies",
    186: "Training",                   187: "Fluoride treatment",
    390: "Mental Health Specialty - Individual Therapy",
    386: "Chemotherapy Drugs",         387: "Upgrades",
    403: "Psychiatric - Individual Therapy",
    408: "Mental Health Specialty - Group Therapy",
    412: "Psychiatric - Group Therapy",
    495: "Non-routine services",       496: "Diagnostic services",
    497: "Restorative services",       498: "Endodontics",
    499: "Periodontics",               500: "Extractions",
    501: "Prosthodontics, other services",
    581: "Routine Services",
}

NETWORK_TYPE_MAP = {
    0: (0, "N/A"),   1: (1, "InNetwork"),
    2: (2, "OutOfNetwork"),  3: (3, "General"),  4: (4, "NA"),
}

print("✅ Libraries and configuration ready")

---
## Cell 3 — Parse the JSON → structured tasks

Reads `input_json` (from Cell 1) and produces:
- **`pbp_context`** — PBP rows indexed by `(header, category)` for fast lookup  
- **`tasks`** — one `BenefitTask` per unique `(BenefitId, TierType)` with its template and resolved token values

In [ ]:
# ── Data classes ──────────────────────────────────────────────────────────────

@dataclass
class BenefitTask:
    """
    One output row — groups all rule rows for a (BenefitId, TierType) pair.
    token_values maps every {{token}} found in the templates to its resolved value.
    """
    benefit_id:     int
    benefit_name:   str
    network_type:   int
    tier_type:      int           # serviceTypeID
    long_template:  str           # DynamicLongValue  → benefitdesc
    tiny_template:  str           # DynamicTinyValue  → tinyDescription
    token_values:   Dict[str, str] = field(default_factory=dict)
    is_static:      bool = False  # True = no LLM needed (HERE / hardcoded)


@dataclass
class OutputRow:
    planid:           str
    plantypeid:       str
    benefitid:        int
    benefitname:      str
    coveragetypeid:   int
    coveragetypedesc: str
    serviceTypeID:    int
    serviceTypeDesc:  str
    benefitdesc:      str
    tinyDescription:  str


# ── Derive plan metadata from PBP rows ────────────────────────────────────────

def derive_plan_meta(pbp_rows: list) -> dict:
    """Extract planid, plan_type from the raw PBP row list."""
    file_name = pbp_rows[0]["FileName"].strip() if pbp_rows else ""

    remainder = file_name[file_name.index('_') + 1:] if '_' in file_name else file_name
    parts     = remainder.split('-')
    contract  = parts[0] if len(parts) > 0 else ""
    plan      = parts[1] if len(parts) > 1 else ""
    seg       = int(parts[2]) if len(parts) > 2 and parts[2].isdigit() else 0
    planid    = f"{CARRIER_PREFIX}_{contract}_{plan}_{seg}"

    plan_type = "HMO"
    for row in pbp_rows:
        if row.get("header") == "Plan Characteristics" and row.get("field") == "Plan Type":
            plan_type = row["value"].strip()
            break

    return {"file_name": file_name, "planid": planid, "plan_type": plan_type}


# ── Index PBP rows for context lookup ─────────────────────────────────────────

def index_pbp(pbp_rows: list) -> Dict[str, list]:
    """
    Returns a dict keyed by lowercase category string → list of {field, value} dicts.
    Used by the LLM prompt to supply extra PBP context per benefit.
    """
    idx = defaultdict(list)
    for row in pbp_rows:
        key = row.get("category", "").strip().lower()
        idx[key].append({"field": row.get("field", ""), "value": row.get("value", "")})
    return dict(idx)


# ── Build BenefitTasks from rules rows ────────────────────────────────────────

def build_tasks(rules_rows: list) -> List[BenefitTask]:
    """
    Groups rule rows by (BenefitId, TierType).

    Each group shares one template set (DynamicLongValue / DynamicTinyValue).
    Multiple rows in the same group each contribute one token→value pair
    (field col = token name, value col = resolved value).

    Static detection:
      - field == 'HERE' and value == 'HERE'  → template IS the output, copy as-is
      - field == 'DEFAULT' and value == 'DEFAULT' → also static (pre-filled long/tiny)
    """
    groups: Dict[Tuple[int, int], List[dict]] = defaultdict(list)
    for row in rules_rows:
        key = (int(row["BenefitId"]), int(row["TierType"]))
        groups[key].append(row)

    tasks = []
    for (bid, tier), rows in groups.items():
        first = rows[0]

        # Detect static: field/value both sentinel strings
        is_static = (
            first["field"].strip().upper()  in ("HERE", "DEFAULT") and
            first["value"].strip().upper() in ("HERE", "DEFAULT")
        )

        # Collect token → resolved-value pairs from all rows in this group
        token_values = {}
        for row in rows:
            fn = row["field"].strip()
            vl = row["value"].strip()
            if fn and fn.upper() not in ("HERE", "DEFAULT"):
                token_values[fn] = vl

        tasks.append(BenefitTask(
            benefit_id    = bid,
            benefit_name  = first["BenefitName"].strip(),
            network_type  = int(first["NetworkType"]),
            tier_type     = tier,
            long_template = first["DynamicLongValue"].strip(),
            tiny_template = first["DynamicTinyValue"].strip(),
            token_values  = token_values,
            is_static     = is_static,
        ))

    tasks.sort(key=lambda t: (t.benefit_id, t.tier_type))
    return tasks


# ── Run parsing ───────────────────────────────────────────────────────────────

meta      = derive_plan_meta(input_json["pbp"])
pbp_index = index_pbp(input_json["pbp"])
tasks     = build_tasks(input_json["benefit_rules"])

static_count = sum(1 for t in tasks if t.is_static)
llm_count    = len(tasks) - static_count

print(f"✅ Parsing complete")
print(f"   Plan ID:           {meta['planid']}")
print(f"   Plan Type:         {meta['plan_type']}")
print(f"   Benefit tasks:     {len(tasks)}")
print(f"   Static (no LLM):   {static_count}")
print(f"   LLM tasks:         {llm_count}")
print()

# Preview all tasks
print(f"  {'BenefitId':>10}  {'TierType':>8}  {'Static':>6}  Benefit Name")
print("  " + "-" * 75)
for t in tasks:
    svc = SERVICE_TYPE_MAP.get(t.tier_type, str(t.tier_type))
    print(f"  {t.benefit_id:>10}  {svc[:8]:>8}  {'YES' if t.is_static else 'LLM':>6}  "
          f"{t.benefit_name}")

---
## Cell 4 — Build the LangChain chain
System prompt encodes all substitution rules derived from the SQL stored procedures.  
Human prompt supplies the template + resolved token values for each task.

In [ ]:
SYSTEM_PROMPT = """\
You are a Medicare plan benefits description generator.

You receive a DynamicLongValue template (→ benefitdesc) and a DynamicTinyValue
template (→ tinyDescription), each containing {{token}} placeholders, plus a
dictionary that maps every token to its resolved value.

Your only job is to substitute every {{token}} with its resolved value and return
the two completed strings.

════════════════════════════════════════════════════════
SUBSTITUTION RULES  (derived from ParseBenefitRules SQL)
════════════════════════════════════════════════════════

1. DOLLAR AMOUNT CLEANUP
   • Strip trailing .00 from all dollar values:  $0.00 → $0,  $250.00 → $250
   • Preserve commas in large amounts:  $2,700 stays $2,700
   • Do not alter percentage values:  18%  stays  18%

2. HTML — benefitdesc only
   • The template already contains <b> tags around placeholders — keep them.
   • Preserve all <br> tags exactly where they appear.
   • Do NOT add or remove any HTML not already in the template.

3. tinyDescription — plain text only
   • Strip every HTML tag: <b>$25</b> copay → $25
   • If the tiny template is a bare {{token}}, resolve it to its plain value.
   • If tiny template is '{{Minimum copayment}} - {{Maximum copayment}}',
     output '$X - $Y' (e.g. '$0 - $25').  Keep the dash.

4. PERIODICITY normalisation
   • 'Every Year' or 'Every year'        → 'every year'
   • 'Every 2 Years'                     → 'every two years'
   • 'Every 3 Years' / 'Every three years' → 'every three years'
   • 'Every 3 Months'                    → 'every three months'
   • 'Every Quarter'                     → 'every quarter'
   • 'Every 6 Months'                    → 'every six months'
   • Any other value                     → lowercase as-is

5. FORMULARY EXCEPTION TIER
   When the token value is 'Tier 4 - Non-Preferred Drug',
   rewrite the dash as a colon: 'Tier 4: Non-Preferred Drug'

6. DRUG DEDUCTIBLE TIER LIST
   When a token value is a semicolon-separated tier list like
   'Tier 1 - Preferred Generic; Tier 2 - Generic; Tier 3 - Preferred Brand',
   shorten it to 'Tier 1, Tier 2, and Tier 3' in the output.

7. PRE-COMPUTED GROUP TOKENS  ({{days-LongValue}}, {{vision-TinyValue}}, etc.)
   These are complete strings already in token_values — substitute them verbatim.

8. STATIC / HERE TASKS
   If there are no {{tokens}} in either template, return the templates as-is.

════════════════════════════════════════════════════════
OUTPUT — return ONLY this JSON, nothing else:
{
  "benefitdesc": "<filled HTML string>",
  "tinyDescription": "<filled plain-text string>"
}
"""

HUMAN_TEMPLATE = """\
Benefit ID:   {benefit_id}
Benefit Name: {benefit_name}
Network:      {network_desc}
Service Type: {service_type_desc}

DynamicLongValue template:
{long_template}

DynamicTinyValue template:
{tiny_template}

Resolved token values:
{token_values_str}
"""

llm = AzureChatOpenAI(
    azure_endpoint   = AZURE_OPENAI_ENDPOINT,
    api_key          = AZURE_OPENAI_API_KEY,
    azure_deployment = AZURE_OPENAI_DEPLOYMENT,
    api_version      = AZURE_OPENAI_API_VERSION,
    temperature      = 0.0,
    max_tokens       = 512,
)

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("human",  HUMAN_TEMPLATE),
])

chain = prompt | llm | StrOutputParser()

print(f"✅ Chain ready — model: {AZURE_OPENAI_DEPLOYMENT}")

---
## Cell 5 — Helper utilities

In [ ]:
def strip_html(text: str) -> str:
    return re.sub(r"<[^>]+>", "", text).strip()


def parse_llm_output(raw: str) -> Tuple[str, str]:
    """Parse JSON from LLM response; falls back gracefully."""
    text = raw.strip()
    # Strip markdown fences if present
    text = re.sub(r"^```[a-z]*\n?", "", text, flags=re.MULTILINE)
    text = re.sub(r"\n?```$",       "", text, flags=re.MULTILINE)
    try:
        parsed = json.loads(text.strip())
        return parsed["benefitdesc"].strip(), parsed["tinyDescription"].strip()
    except Exception:
        # Regex fallback
        bd = re.search(r'"benefitdesc"\s*:\s*"(.*?)"(?=\s*,|\s*})', text, re.DOTALL)
        td = re.search(r'"tinyDescription"\s*:\s*"(.*?)"(?=\s*})',   text, re.DOTALL)
        return (
            bd.group(1).strip() if bd else "See Brochure",
            td.group(1).strip() if td else "See Brochure",
        )


def resolve_static(task: BenefitTask) -> Tuple[str, str]:
    """Return static (HERE / DEFAULT) task values without calling the LLM."""
    return task.long_template, strip_html(task.tiny_template) or strip_html(task.long_template)


def format_token_values(token_values: dict) -> str:
    """Format token dict as readable string for the human prompt."""
    if not token_values:
        return "  (none — template contains no substitution tokens)"
    return "\n".join(f"  {{{{ {k} }}}} = {v}" for k, v in token_values.items())


print("✅ Helper utilities defined")

---
## Cell 6 — Process all tasks → output rows
Loops over every `BenefitTask`.  
Static tasks are resolved instantly; LLM tasks call Azure OpenAI.

In [ ]:
def process_tasks(
    tasks:     List[BenefitTask],
    meta:      dict,
    chain,
    dry_run:   bool = False,
) -> List[OutputRow]:
    """
    Process every BenefitTask and return a list of OutputRow objects.

    Args:
        tasks:   Built by build_tasks() in Cell 3
        meta:    Plan metadata dict from derive_plan_meta()
        chain:   LangChain chain (prompt | llm | parser)
        dry_run: Skip LLM calls — return unfilled templates (for testing)
    """
    planid    = meta["planid"]
    plan_type = meta["plan_type"]
    rows      = []
    n         = len(tasks)

    for i, task in enumerate(tasks, 1):
        cov_id, cov_desc = NETWORK_TYPE_MAP.get(task.network_type, (task.network_type, str(task.network_type)))
        svc_desc         = SERVICE_TYPE_MAP.get(task.tier_type, str(task.tier_type))
        label            = f"[{i:>3}/{n}]  BenefitId={task.benefit_id:<5}  {task.benefit_name[:38]:<38}  svc={svc_desc[:18]}"

        # ── Static path (no LLM) ─────────────────────────────────────────────
        if task.is_static:
            long_val, tiny_val = resolve_static(task)
            print(f"  STATIC   {label}  → {tiny_val[:30]}")

        # ── Dry run (no LLM) ─────────────────────────────────────────────────
        elif dry_run:
            long_val = task.long_template
            tiny_val = task.tiny_template
            print(f"  DRY RUN  {label}")

        # ── LLM path ─────────────────────────────────────────────────────────
        else:
            try:
                raw = chain.invoke({
                    "benefit_id":        task.benefit_id,
                    "benefit_name":      task.benefit_name,
                    "network_desc":      cov_desc,
                    "service_type_desc": svc_desc,
                    "long_template":     task.long_template,
                    "tiny_template":     task.tiny_template,
                    "token_values_str":  format_token_values(task.token_values),
                })
                long_val, tiny_val = parse_llm_output(raw)
                print(f"  LLM ✓    {label}  → {tiny_val[:30]}")
            except Exception as e:
                print(f"  ERROR    {label}  !! {e}")
                long_val, tiny_val = "See Brochure", "See Brochure"

        rows.append(OutputRow(
            planid           = planid,
            plantypeid       = plan_type,
            benefitid        = task.benefit_id,
            benefitname      = task.benefit_name,
            coveragetypeid   = cov_id,
            coveragetypedesc = cov_desc,
            serviceTypeID    = task.tier_type,
            serviceTypeDesc  = svc_desc,
            benefitdesc      = long_val,
            tinyDescription  = tiny_val,
        ))

    return rows


# ── DRY RUN first — verify routing without spending LLM tokens ────────────────
print("DRY RUN (no Azure OpenAI calls)\n")
dry_rows = process_tasks(tasks, meta, chain, dry_run=True)
print(f"\n✅ Dry run complete — {len(dry_rows)} rows")

---
## Cell 7 — Live run (calls Azure OpenAI)

In [ ]:
print("LIVE RUN — calling Azure OpenAI\n")
output_rows = process_tasks(tasks, meta, chain, dry_run=False)
print(f"\n✅ {len(output_rows)} rows generated")

---
## Cell 8 — Results, export & validation

In [ ]:
OUTPUT_COLUMNS = [
    "planid", "plantypeid", "benefitid", "benefitname",
    "coveragetypeid", "coveragetypedesc",
    "serviceTypeID", "serviceTypeDesc",
    "benefitdesc", "tinyDescription",
]

df = (
    pd.DataFrame([r.__dict__ for r in output_rows])
    [OUTPUT_COLUMNS]
    .sort_values(["benefitid", "serviceTypeID"])
    .reset_index(drop=True)
)

# ── Save CSV ──────────────────────────────────────────────────────────────────
CSV_OUT = f"output_benefits_loadid{input_json['load_id']}.csv"
df.to_csv(CSV_OUT, index=False)
print(f"✅ Saved → {CSV_OUT}  ({len(df)} rows)")

# ── Preview ───────────────────────────────────────────────────────────────────
print()
df[["benefitid", "benefitname", "serviceTypeDesc", "tinyDescription"]].head(20)

In [ ]:
# ── Validate against expected output (optional) ───────────────────────────────
EXPECTED_PATH = "example_final_result.txt"

try:
    exp = pd.read_csv(EXPECTED_PATH)
    merged = exp.merge(
        df, on=["benefitid", "serviceTypeID"],
        suffixes=("_exp", "_act"), how="outer"
    ).dropna(subset=["tinyDescription_exp", "tinyDescription_act"])

    merged["match"] = (
        merged["tinyDescription_exp"].str.strip() ==
        merged["tinyDescription_act"].str.strip()
    )
    pct = merged["match"].mean() * 100
    print(f"tinyDescription accuracy: {merged['match'].sum()}/{len(merged)} ({pct:.0f}%)")

    mismatches = merged[~merged["match"]][["benefitid", "benefitname_exp",
                                            "serviceTypeID", "tinyDescription_exp",
                                            "tinyDescription_act"]]
    if len(mismatches):
        print("\nMismatches:")
        for _, r in mismatches.iterrows():
            print(f"  [{r.benefitid}] {r.benefitname_exp}  svc={r.serviceTypeID}")
            print(f"    expected: {r.tinyDescription_exp}")
            print(f"    actual:   {r.tinyDescription_act}")
    else:
        print("✅ All tinyDescriptions match!")

except FileNotFoundError:
    print(f"Skipping validation — '{EXPECTED_PATH}' not found")

In [ ]:
# ── Inspect one benefit in full ───────────────────────────────────────────────
INSPECT_ID = 620   # change to any benefitid

for r in output_rows:
    if r.benefitid == INSPECT_ID:
        print(f"[{r.benefitid}] {r.benefitname}  svc={r.serviceTypeDesc}")
        print(f"benefitdesc:\n  {r.benefitdesc}")
        print(f"tinyDescription: {r.tinyDescription}")
        print("-" * 60)